In [1]:
from langchain_core.runnables import RunnableLambda, RunnableBranch, RunnableParallel, branch
from pydantic import BaseModel, Field

from  langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from openai import OpenAI
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from typing import Literal, TypedDict, List
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END
from IPython.display import Image,display, Markdown
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper



# This function will load all the variables from the .env file and will
# make them available in the os.environ dictionary (env variables)
load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()

class llm_schema(BaseModel):
    funny_flag: Literal["funny","not funny"] = Field(description="Whether the joke is funny or not")
    feedback: str = Field(description="Feedback on the joke")

llm_with_schema = llm_openai.with_structured_output(llm_schema)


class graph_schema(TypedDict):

    topic: str
    joke: str
    funny_flag: str
    feedback: str
    max_iterations: int

def generate_node(state: graph_schema) -> graph_schema:


    topic = state['topic']

    if state['feedback']: # Not first iteration, generate a joke based on feedback
        response = llm_openai.invoke(f"Please modify the following joke {state['joke']} based on the following feedback: {state['feedback']}")
    else: # First iteration, generate a joke
        response = llm_openai.invoke("Create only one joke about the following topic: " + topic)


    state['joke'] = response.content
    return state

def evaluate_node(state: graph_schema) -> graph_schema:

    joke = state['joke']
    iteration = state['max_iterations']

    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a comedy critic. Your job is to evaluate the following joke and provide feedback on how to make it funnier."),
        ("user", f"Evaluate the following joke: {joke}\nRespond with 'funny' or 'not_funny' and provide feedback if it's not funny.")
    ])

    chain = prompt | llm_with_schema
    response = chain.invoke({"joke": joke})

    state['funny_flag'] = response.funny_flag
    state['feedback'] = response.feedback
    state['max_iterations'] = iteration + 1

    return state

def check_iteration(state: graph_schema) -> str:

    iteration = state['max_iterations']

    if iteration <= 5 and state['funny_flag'] != "funny":
        return "evaluate_node"
    else:
        return "end"






graph=StateGraph(graph_schema)

graph.add_node("generate_node", generate_node)
graph.add_node("evaluate_node", evaluate_node)

graph.add_edge(START, "generate_node")
graph.add_conditional_edges("generate_node", check_iteration, {"evaluate_node": "evaluate_node", "end": END})
graph.add_edge("evaluate_node", "generate_node")
graph.add_edge("generate_node", END)

f_graph=graph.compile()

# Display Mermaid text instead of trying to render as PNG
print(f_graph.get_graph().draw_mermaid())

for chunk in f_graph.stream(
    {
        "topic": "Cars",
        "joke": "",
        "funny_flag": "",
        "feedback": "",
        "max_iterations": 0
    },
    stream_mode="updates"
):
    print(chunk)

C:\Users\Ujjwal_Manna\AppData\Local\Temp\ipykernel_15700\1798805683.py:21: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun
C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pyppeteer\us_keyboard_layout.py:73: SyntaxWarning: invalid escape sequence '\('
  'Digit9': {'keyCode': 57, 'code': 'Digit9', 'shiftKey': '\(', 'key': '9'},
C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pyppeteer\us_keyboard_layout.py:143: SyntaxWarning: invalid escape sequence '\<'
  'Comma': {'keyCode': 188, 'code': 'Comma', 'shiftKey': '\<', 'key': ','},
C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pyppeteer\us_keyboard_layout.py:247: SyntaxWarning: invalid escape sequence '\<'
  '<': {'keyCode': 188, 'key': '\<', 'code': '

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	generate_node(generate_node)
	evaluate_node(evaluate_node)
	__end__([<p>__end__</p>]):::last
	__start__ --> generate_node;
	evaluate_node --> generate_node;
	generate_node -. &nbsp;end&nbsp; .-> __end__;
	generate_node -.-> evaluate_node;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

{'generate_node': {'topic': 'Cars', 'joke': 'Why did the car apply for a job? Because it wanted to get a "drive" in its career!', 'funny_flag': '', 'feedback': '', 'max_iterations': 0}}


C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=llm_schema(funny_flag='no...ayered and surprising.'), input_type=llm_schema])
  return self.__pydantic_serializer__.to_python(


{'evaluate_node': {'topic': 'Cars', 'joke': 'Why did the car apply for a job? Because it wanted to get a "drive" in its career!', 'funny_flag': 'not funny', 'feedback': 'The joke leans on a simple car‑related pun ("drive"), but it feels a bit flat because the punchline is predictable and doesn\'t add a surprising twist. To boost the humor:\n\n1. **Add a surprising element** – introduce an unexpected reason or a double‑meaning that catches the listener off guard.\n2. **Play with car terminology** – mix in other car‑related words (e.g., "gear," "shift," "horsepower") for a richer wordplay.\n3. **Set up a stronger contrast** – juxtapose the car’s literal abilities with a human‑like career ambition.\n\n**Revised version example:**\n> *Why did the sedan go to a job interview?* \n> *Because it heard the company was looking for someone with a lot of “traction” and wanted to “shift” into a higher gear!* \n\nThis version adds a second pun (traction/shift) and a clearer image of the car trying t

C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=llm_schema(funny_flag='no...a genuinely funny one.'), input_type=llm_schema])
  return self.__pydantic_serializer__.to_python(


{'evaluate_node': {'topic': 'Cars', 'joke': '**Why did the car apply for a job?**  \nBecause it heard the company needed someone with real *horsepower* to *shift* the business into a higher *gear*—and it was tired of staying stuck in neutral while everyone else was getting a promotion!', 'funny_flag': 'not funny', 'feedback': 'The premise— a car applying for a job—has potential, but the execution gets bogged down in a laundry list of car‑related puns. Here’s how you can tighten it up and boost the laugh factor:\n\n1. **Keep it concise** – A joke lands better when the punchline arrives quickly. Trim the setup to one clear line and let the wordplay do the heavy lifting.\n\n2. **Focus on a single, strong pun** – Throwing in *horsepower*, *shift*, *gear*, and *neutral* all at once dilutes the impact. Pick the one that feels freshest and build the punch around it.\n\n3. **Add a surprise twist** – The humor spikes when the listener expects one thing and gets another. For example, play on the

C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=llm_schema(funny_flag='funny', feedback=''), input_type=llm_schema])
  return self.__pydantic_serializer__.to_python(


{'evaluate_node': {'topic': 'Cars', 'joke': '**Re‑worked joke (tight, single‑pun, surprise twist)**  \n\n> **Why did the car apply for a job?**  \n> Because it was tired of being stuck in *neutral* and finally wanted a promotion to *first gear*.\n\n---\n\n### Why this lands better\n\n| Feedback point | How the new version addresses it |\n|----------------|-----------------------------------|\n| **Keep it concise** | The setup is a single, clear question; the punchline is one sentence. |\n| **Focus on a single, strong pun** | The joke leans only on the “neutral → first gear” pun, avoiding the clutter of horsepower, shift, etc. |\n| **Add a surprise twist** | Listeners expect a typical “horsepower” line, but the twist is the car’s “career‑ladder” metaphor—moving from neutral (no progress) to first gear (a real step forward). |\n| **Use a relatable job‑search metaphor** | “Stuck in neutral” mirrors the feeling of being stuck in a dead‑end job; “promotion to first gear” feels like that cov